# 07 — OE Image Inversion with Dask

This notebook demonstrates OE-based image inversion using `dask_oe_engine`.  
It uses the same Sentinel-2 Wadden Sea dataset as NB04, but replaces the
lmfit pixel-by-pixel loop with JAX-vmapped Gauss-Newton OE — providing
not just retrieved parameters but also **uncertainty maps** (posterior σ),
**averaging kernel maps** (how data-driven each pixel is), and a
**chi-squared map** (goodness of fit).

Key differences vs NB04:
- Forward model: `albert_mobley_jax` (JAX, exact Jacobians) instead of `albert_mobley` (lmfit)
- Retrieved params: C_0, C_Y, C_Mie, zB (vs. zB only in NB04)
- Bottom: per-tile mean of low-tide image, rebuilt per Dask tile (same spirit as NB04)
- Outputs: xarray Dataset with x_hat, σ, A_diag, chi² per pixel

In [ ]:
import os
import numpy as np
import xarray as xr
import lmfit
import dask
import matplotlib.pyplot as plt
import jax.numpy as jnp

from bio_optics.inversion import oe_engine, dask_oe_engine
from bio_optics.reflectance import albert_mobley_jax

## Get data

Same Sentinel-2 Wadden Sea scene as NB04: two time steps at the same location.
Time 0 = low tide (benthic reflectance visible), Time 1 = high tide (water-leaving Rrs).

In [ ]:
dataset = xr.open_dataset(os.path.join(os.getcwd(), 'example_data/S2L2A_example.nc'))
dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dataset[['B04', 'B03', 'B02']].isel(time=0).to_array().plot.imshow(robust=True, ax=axes[0])
axes[0].set_title('Low tide (benthic albedo visible)')
dataset[['B04', 'B03', 'B02']].isel(time=1).to_array().plot.imshow(robust=True, ax=axes[1])
axes[1].set_title('High tide (water-leaving Rrs — what we invert)')
plt.tight_layout()
plt.show()

## Prepare arrays

- `Rrs_image`: high-tide reflectance divided by π → Rrs [sr⁻¹], shape (n_rows, n_cols, n_obs)
- `albedo_image`: low-tide reflectance **not** divided by π → albedo [dimensionless],
  used as per-tile bottom input (same convention as NB04).

In [ ]:
wavelengths = np.array([490, 560, 665, 705, 740, 783, 842, 865])

band_vars = list(dataset.data_vars)[1:]   # skip the first variable (time or similar)

# --- detect spatial dimension names and order from the dataset ---------------
# The dataset may store bands as (x, y) or (y, x); we must preserve that order
# when stacking so that the reshape after inversion is consistent.
ref_band = dataset.isel(time=1)[band_vars[0]]
spatial_dim_names = list(ref_band.dims)          # e.g. ['x', 'y'] or ['y', 'x']
print('Spatial dims in dataset:', spatial_dim_names)

# Extract spatial coordinates for correct plot orientation later
spatial_coords = {d: dataset[d].values for d in spatial_dim_names if d in dataset.coords}

# --- stack bands (bands last, spatial dims in dataset-native order) ----------
Rrs_image = np.stack(
    [dataset.isel(time=1)[v].values for v in band_vars], axis=-1
) / np.pi
print('Rrs_image shape:', Rrs_image.shape)    # (dim0, dim1, n_obs)

albedo_image = np.stack(
    [dataset.isel(time=0)[v].values for v in band_vars], axis=-1
)
print('albedo_image shape:', albedo_image.shape)

## Precompute

Run the spectral lookup table resampling once. We do **not** insert the bottom
albedo here — it will be inserted per tile during inversion, giving each tile its
own representative bottom spectrum (tile mean of the low-tide image).

In [ ]:
pre_base = albert_mobley_jax.precompute(wavelengths)
print('precomputed keys:', list(pre_base.keys()))

## Set up parameters

All parameters needed by `albert_mobley_jax`. We retrieve **C_0, C_Y, C_Mie, zB**
with log-transforms (lognormal prior); everything else is fixed.

Bottom configuration: `f_0=1` (100% first bottom type), `B_0=1/π` (Lambertian),
so `Rrs_b = bottom_albedo / π` — same convention as NB04.

In [ ]:
params = lmfit.Parameters()

# --- Geometry ---
params.add('theta_sun',  value=np.radians(30), vary=False)
params.add('theta_view', value=np.radians(0),  vary=False)
params.add('n1',         value=1.0,            vary=False)
params.add('n2',         value=1.33,           vary=False)
params.add('kappa_0',    value=1.0546,         vary=False)

# --- Water constituents (retrieved) ---
params.add('C_0',   value=0.5,  vary=True)   # phytoplankton [mg/m3]
params.add('C_Y',   value=0.1,  vary=True)   # CDOM absorption at 440 nm [m-1]
params.add('C_Mie', value=0.1,  vary=True)   # Mie particles [g/m3]

# --- Water constituents (fixed) ---
params.add('C_1',   value=0.0,  vary=False)
params.add('C_2',   value=0.0,  vary=False)
params.add('C_3',   value=0.0,  vary=False)
params.add('C_4',   value=0.0,  vary=False)
params.add('C_5',   value=0.0,  vary=False)
params.add('C_X',   value=0.0,  vary=False)  # type-I (spectrally flat) particles

# --- IOP parameters (fixed) ---
params.add('S',                   value=0.014,  vary=False)
params.add('S_NAP',               value=0.011,  vary=False)
params.add('lambda_0',            value=440.0,  vary=False)
params.add('K',                   value=0.0,    vary=False)
params.add('T_W',                 value=18.0,   vary=False)
params.add('T_W_0',               value=20.0,   vary=False)
params.add('a_NAP_spec_lambda_0', value=0.041,  vary=False)
params.add('bb_phy_spec',         value=0.0010, vary=False)
params.add('bb_Mie_spec',         value=0.0042, vary=False)
params.add('bb_X_spec',           value=0.0086, vary=False)
params.add('lambda_S',            value=500.0,  vary=False)
params.add('n',                   value=-1.0,   vary=False)

# --- Bottom (fixed: 100% first type = measured benthic albedo, inserted per tile) ---
params.add('f_0', value=1.0,      vary=False)
params.add('f_1', value=0.0,      vary=False)
params.add('f_2', value=0.0,      vary=False)
params.add('f_3', value=0.0,      vary=False)
params.add('f_4', value=0.0,      vary=False)
params.add('f_5', value=0.0,      vary=False)
params.add('B_0', value=1/np.pi,  vary=False)
params.add('B_1', value=1/np.pi,  vary=False)
params.add('B_2', value=1/np.pi,  vary=False)
params.add('B_3', value=1/np.pi,  vary=False)
params.add('B_4', value=1/np.pi,  vary=False)
params.add('B_5', value=1/np.pi,  vary=False)

# --- Depth (retrieved) ---
params.add('zB', value=2.0, vary=True)

print('Free parameters:', [n for n in params if params[n].vary])

## Prior and noise

- `sigma_a`: in **retrieval space** — for log-params this means relative (fractional)
  uncertainty: 1.0 = ±100% (factor-of-e), 0.69 ≈ factor-of-two.
- `noise`: measurement uncertainty [sr⁻¹]. 0.001 sr⁻¹ is a conservative estimate
  for Sentinel-2 L2A over water (dominated by atmospheric correction residuals).

In [ ]:
log_params = ['C_0', 'C_Y', 'C_Mie', 'zB']

sigma_a = {
    'C_0':   1.0,   # ±100% relative uncertainty
    'C_Y':   1.0,
    'C_Mie': 1.0,
    'zB':    0.7,   # ≈ factor-of-two depth uncertainty
}

noise     = 0.001   # measurement std [sr-1]
n_iter    = 15
tile_size = 4096    # pixels per tile

## Run inversion — per-tile bottom

For each tile we:
1. Extract the tile's low-tide albedo spectra and compute the **tile mean** bottom
2. Insert it as the first bottom type in a tile-specific `precomputed` dict
3. Build `f_vec` and `InversionSetup` for that tile
4. Dispatch a Dask task calling `invert_tile()`

JAX retraces on the first tile (and whenever the tile's `f_fit` object changes —
once per tile here).  Each trace/compile takes a few seconds; subsequent tiles on
the same worker reuse XLA compiled code if the f_fit is identical.

> **Note:** True per-pixel bottom (not per-tile mean) would require expressing
> `R_b_i` as a vmapped JAX array, which needs a small forward-model change.
> Per-tile mean is a good approximation when tiles are spatially coherent.

In [ ]:
all_names = list(params.keys())
n_dim0, n_dim1, n_obs = Rrs_image.shape    # native dataset spatial shape
Rrs_flat    = Rrs_image.reshape(-1, n_obs)
albedo_flat = albedo_image.reshape(-1, n_obs)
n_pixels    = len(Rrs_flat)

delayed_tasks = []

for start in range(0, n_pixels, tile_size):
    end = min(start + tile_size, n_pixels)

    # --- per-tile bottom: mean of this tile's low-tide spectra ---------------
    bottom_tile = np.nanmean(albedo_flat[start:end], axis=0)   # (n_obs,)

    # --- build precomputed dict with this tile's bottom ----------------------
    pre_tile   = dict(pre_base)
    R_b_i_tile = np.array(pre_base['R_b_i'])                  # (n_obs, 6)
    R_b_i_tile[:, 0] = bottom_tile
    pre_tile['R_b_i'] = jnp.array(R_b_i_tile)

    # --- build f_vec and InversionSetup for this tile ------------------------
    f_vec_tile  = albert_mobley_jax.make_forward_vec(all_names, pre_tile)
    setup_tile  = oe_engine.build_inversion(
        params, f_vec_tile, sigma_a, log_params=log_params
    )

    # --- dispatch Dask task --------------------------------------------------
    task = dask.delayed(dask_oe_engine.invert_tile)(
        Rrs_flat[start:end],
        setup_tile.f_fit,
        np.array(setup_tile.x_a),
        np.array(setup_tile.S_a_inv),
        np.array(setup_tile.log_mask),
        noise,
        None,    # weights
        n_iter,
        0.0,     # lm_damping
        False,   # store_y_hat
    )
    delayed_tasks.append(task)

print(f'{len(delayed_tasks)} tiles → running...')
tile_results = dask.compute(*delayed_tasks, scheduler='synchronous')
print('Done.')

## Assemble results

In [ ]:
n_fit = len(setup_tile.fit_names)

x_hat_all  = np.concatenate([r[0] for r in tile_results], axis=0).reshape(n_dim0, n_dim1, n_fit)
sigma_all  = np.concatenate([r[1] for r in tile_results], axis=0).reshape(n_dim0, n_dim1, n_fit)
A_diag_all = np.concatenate([r[2] for r in tile_results], axis=0).reshape(n_dim0, n_dim1, n_fit)
chi2_all   = np.concatenate([r[3] for r in tile_results], axis=0).reshape(n_dim0, n_dim1)

results = {
    'x_hat':     x_hat_all,
    'sigma':     sigma_all,
    'A_diag':    A_diag_all,
    'chi2':      chi2_all,
    'fit_names': setup_tile.fit_names,
}

# Pass native spatial dim names + original coordinates so plots are oriented
# exactly like the source dataset (no transposition or mirroring artefacts).
ds = dask_oe_engine.to_dataset(
    results,
    spatial_dims=tuple(spatial_dim_names),
    coords=spatial_coords,
)
ds

## Goodness of fit — χ²

χ² ≈ 1 means the residuals match the assumed noise level.  
χ² >> 1 flags model–data mismatch: cloud shadows, emergent vegetation, or
optically deep pixels where the shallow-water model is inappropriate.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ds['chi2'].plot(ax=ax, vmin=0, vmax=5, cmap='RdYlGn_r')
ax.set_title('χ² (goodness of fit — ideal ≈ 1)')
plt.tight_layout()
plt.show()

print(f'Median χ²: {float(np.nanmedian(chi2_all)):.2f}')
print(f'Pixels with χ² > 3: {(chi2_all > 3).sum()} / {chi2_all.size}')

## Retrieved parameters

In [ ]:
param_labels = {
    'C_0':   ('Chlorophyll C₀', 'mg m⁻³', 'Greens'),
    'C_Y':   ('CDOM C_Y',       'm⁻¹',    'YlOrBr'),
    'C_Mie': ('SPM C_Mie',      'g m⁻³',  'Oranges'),
    'zB':    ('Depth z_B',      'm',       'Blues_r'),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (param, (label, unit, cmap)) in zip(axes.flat, param_labels.items()):
    ds['x_hat'].sel(param=param).plot(ax=ax, cmap=cmap, robust=True)
    ax.set_title(f'{label} [{unit}]')

plt.suptitle('Retrieved parameters (physical space)', y=1.01)
plt.tight_layout()
plt.show()

## Posterior uncertainty — σ

σ is the posterior standard deviation in **physical space** (delta-method for log-params).  
Compare σ to the prior to see how much the data constrained each parameter.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (param, (label, unit, _)) in zip(axes.flat, param_labels.items()):
    ds['sigma'].sel(param=param).plot(ax=ax, cmap='Purples', robust=True)
    ax.set_title(f'σ({label}) [{unit}]')

plt.suptitle('Posterior uncertainty σ (physical space)', y=1.01)
plt.tight_layout()
plt.show()

## Averaging kernel diagonal — A

A[i,i] ∈ [0, 1]: fraction of the retrieved value that comes from the data
(vs. the prior).  A ≈ 1 = fully data-driven; A ≈ 0 = mostly prior.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (param, (label, _, __)) in zip(axes.flat, param_labels.items()):
    ds['A_diag'].sel(param=param).plot(ax=ax, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'A[{param},{param}] (data fraction)')

plt.suptitle('Averaging kernel diagonal (0 = prior, 1 = data)', y=1.01)
plt.tight_layout()
plt.show()

dfs = A_diag_all.sum(axis=-1)
print(f'Median DFS: {np.nanmedian(dfs):.2f} / {len(setup_tile.fit_names)} parameters')

## Per-parameter summary

In [ ]:
header = f"{'Parameter':<10}  {'median x_hat':>12}  {'median sigma':>12}  {'median A[i,i]':>13}"
print(header)
print('-' * len(header))
for i, name in enumerate(setup_tile.fit_names):
    x_med = np.nanmedian(x_hat_all[..., i])
    s_med = np.nanmedian(sigma_all[..., i])
    a_med = np.nanmedian(A_diag_all[..., i])
    print(f'{name:<10}  {x_med:>12.4f}  {s_med:>12.4f}  {a_med:>13.3f}')